In [ ]:
# Mount Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
! pip install transformers datasets evaluate seqeval -q

In [ ]:
def parse_conll_file(conll_file_path):
    data = []
    tokens = []
    ner_tags = []
    tag2id = {"O": 0, "claim": 1, "premise": 2}

    with open(conll_file_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if line == "":
                if tokens:
                    data.append({'id': str(len(data)), 'ner_tags': ner_tags, 'tokens': tokens})
                    tokens, ner_tags = [], []
            else:
                parts = line.split()
                if len(parts) != 3:
                    continue
                token, _, tag = parts
                tokens.append(token)
                if tag in ["B-claim", "I-claim"]:
                    ner_tags.append(tag2id["claim"])
                elif tag in ["B-premise", "I-premise"]:
                    ner_tags.append(tag2id["premise"])
                elif tag == "O":
                    ner_tags.append(tag2id["O"])

        if tokens:
            data.append({'id': str(len(data)), 'ner_tags': ner_tags, 'tokens': tokens})

    return data

In [ ]:
conll_file_path = "/path/to/your/file"
data = parse_conll_file(conll_file_path)

In [ ]:
all_labels = {label for entry in data for label in entry["ner_tags"]}
unexpected_labels = all_labels - {0, 1, 2}
if unexpected_labels:
    print(f"Warning: Found unexpected label IDs: {unexpected_labels}")

In [ ]:
import random
import numpy as np
import torch
import os
from sklearn.metrics import classification_report
from datasets import Dataset
from transformers import (
    BertTokenizerFast, BertForTokenClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
    AutoModelForTokenClassification
)

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# Set seed function
def set_seed(seed):
    torch.backends.cudnn.deterministic = True
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
set_seed(0)

def split_data(data, train_ratio=0.80, dev_ratio=0.10, test_ratio=0.10):
    assert abs(train_ratio + dev_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"

    # Shuffle
    random.shuffle(data)

    # Calculate sizes
    total_size = len(data)
    train_size = int(train_ratio * total_size)
    dev_size = int(dev_ratio * total_size)
    test_size = total_size - train_size - dev_size

    # Split data
    train_data = data[:train_size]
    dev_data = data[train_size:train_size + dev_size]
    test_data = data[train_size + dev_size:]

    return train_data, dev_data, test_data

In [ ]:
train_data, dev_data, test_data = split_data(data)

In [ ]:
[pos["id"] for pos in train_data]

In [ ]:
def chunk_example(example, max_len=400):
    tokens = example["tokens"]
    tags = example["ner_tags"]

    chunks = []
    for start_idx in range(0, len(tokens), max_len):
        end_idx = start_idx + max_len

        chunk_tokens = tokens[start_idx:end_idx]
        chunk_tags = tags[start_idx:end_idx]

        chunk_example = {
            "id": f"{example['id']}_chunk_{start_idx}",
            "tokens": chunk_tokens,
            "ner_tags": chunk_tags
        }
        chunks.append(chunk_example)

    return chunks

In [ ]:
def chunk_data(data, max_len=400):
    chunked_dataset = []
    for example in data:
        chunked_dataset.extend(chunk_example(example, max_len=max_len))
    return chunked_dataset

chunked_train_data = chunk_data(train_data, max_len=400)
chunked_dev_data   = chunk_data(dev_data, max_len=400)
chunked_test_data  = chunk_data(test_data, max_len=400)

print(f"Train examples (original): {len(train_data)} → chunked: {len(chunked_train_data)}")
print(f"Dev examples (original):   {len(dev_data)} → chunked: {len(chunked_dev_data)}")
print(f"Test examples (original):  {len(test_data)} → chunked: {len(chunked_test_data)}")

Train examples (original): 62 → chunked: 138
Dev examples (original):   7 → chunked: 11
Test examples (original):  9 → chunked: 16


In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-large-uncased")

In [ ]:
# Function for tokenization and label alignment
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        max_length=512,
        padding="max_length",
        truncation=True,
        is_split_into_words=True)

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(example["ner_tags"][word_idx])
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx
    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

In [ ]:
# Apply tokenization and alignment on each split
tokenized_train_data = [tokenize_and_align_labels(example) for example in chunked_train_data]
tokenized_dev_data = [tokenize_and_align_labels(example) for example in chunked_dev_data]
tokenized_test_data = [tokenize_and_align_labels(example) for example in chunked_test_data]

In [ ]:
train_dataset = Dataset.from_list(tokenized_train_data)
dev_dataset = Dataset.from_list(tokenized_dev_data)
test_dataset = Dataset.from_list(tokenized_test_data)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Train

In [ ]:
id2label = {
    0: "O",
    1: "claim",
    2: "premise"
}
label2id = {
    "O": 0,
    "claim": 1,
    "premise": 2
}

In [ ]:
# Evaluation function
def evaluate_model(trainer, eval_dataset, id2label):
    if trainer.state.best_model_checkpoint:
        print(f"Loading best model from: {trainer.state.best_model_checkpoint}")
        trainer.model = AutoModelForTokenClassification.from_pretrained(
            trainer.state.best_model_checkpoint
        ).to(trainer.args.device)

    predictions, labels, _ = trainer.predict(eval_dataset)
    preds = np.argmax(predictions, axis=-1)

    true_labels, pred_labels = [], []
    for i, label in enumerate(labels):
        true_labels.extend([id2label[l] for l in label if l != -100])
        pred_labels.extend([id2label[p] for (p, l) in zip(preds[i], label) if l != -100])

    report = classification_report(true_labels, pred_labels, digits=4, output_dict=True)

    macro_f1 = report["macro avg"]["f1-score"]
    claim_f1 = report["claim"]["f1-score"] if "claim" in report else 0.0
    premise_f1 = report["premise"]["f1-score"] if "premise" in report else 0.0

    print(f"Macro F1: {macro_f1:.4f} | Claim F1: {claim_f1:.4f} | Premise F1: {premise_f1:.4f}")
    return {"macro": macro_f1, "claim": claim_f1, "premise": premise_f1}

In [ ]:
# Loop over seeds
seeds = [42, 123, 2025]
results = []

for seed in seeds:
    print(f"\n Training with seed {seed} \n")
    set_seed(seed)

    model = BertForTokenClassification.from_pretrained(
        "bert-large-uncased", num_labels=3, id2label=id2label, label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        learning_rate=2e-5,
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=10,
        per_device_train_batch_size=5,
        per_device_eval_batch_size=5,
        num_train_epochs=8,
        weight_decay=0.01,
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        seed=seed,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()
    run_scores = evaluate_model(trainer, test_dataset, id2label)
    results.append(run_scores)

In [ ]:
# Summary of results
score_types = ["macro", "claim", "premise"]
summary = {
    score: {
        "mean": np.mean([r[score] for r in results]),
        "std": np.std([r[score] for r in results])
    } for score in score_types
}

print("\n Final Results Across Runs \n")
for i, seed in enumerate(seeds):
    r = results[i]
    print(f"Seed {seed}: Macro F1 = {r['macro']:.4f}, Claim F1 = {r['claim']:.4f}, Premise F1 = {r['premise']:.4f}")

print("\n Averaged Results \n")
for score_type in score_types:
    print(f"{score_type.capitalize()} F1: Mean = {summary[score_type]['mean']:.4f}, Std = {summary[score_type]['std']:.4f}")